<a href="https://colab.research.google.com/github/alscop/ESAA-26-1/blob/main/ESAA_OB_week09_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 딥러닝 파이토치 교과서 5장 p.200-229

## 5.3 전이 학습

이미지넷처럼 아주 큰 데이터셋을 써서 훈련된 모델의 가중치를 가져와 우리가 해결하려는 과제에 맞게 보정해서 사용하는 것.  

네트워크(사전 훈련된 모델): 아주 큰 데이터셋을 시용하여 훈련된 모델.  

-> 비교적 적은 수의 데이터를 가지고도 우리가 원하는 과제 해결 가능  


### 5.3.1 특성 추출 기법

사전 훈련된 모델 가져온 후 마지막에 완전연결층 부분만 새로 생성  

학습할 때는 마지막 완전연결층(이미지 카테고리 결정하는 부분)만 학습, 나머지 계층들은 학습되지 않도록 함

- 합성곱층: 합성곱층 & 풀링층
- 데이터 분류기(완전 연결층): 추출 특성 입력받아 최종적으로 이미지에 대한 클래스를 분류

사전 훈련된 네트워크의 합성곱층(가중치 고정)에 새로운 데이터 통과시키고 -> 그 출력을 분류기에서 훈련시킴(여기에서만 학습)





In [1]:
!pip install opencv-python

In [2]:
# 라이브러리 호출

import os
import time
import copy
import glob
import cv2  # 앞에서 설치한 OpenCV 라이브러리
import shutil

import torch
import torchvision  # 컴퓨터 비전 용도의 패키지
import torchvision.transforms as transforms  # 데이터 전처리를 위해 사용되는 패키지
import torchvision.models as models  # 다양한 파이토치 네트워크를 사용할 수 있도록 도와주는 패키지
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [4]:
!git clone https://github.com/gilbutITbook/080289.git

Cloning into '080289'...
remote: Enumerating objects: 2278, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 2278 (delta 1), reused 0 (delta 0), pack-reused 2273 (from 2)
Receiving objects: 100% (2278/2278), 330.30 MiB | 23.94 MiB/s, done.
Resolving deltas: 100% (13/13), done.
Updating files: 100% (2591/2591), done.


In [5]:
# 이미지 데이터 전처리 방법 정의

data_path =  '080289/chap05/data/catanddog/train'  # 이미지 데이터가 위치한 경로 지정

transform = transforms.Compose(
    [
        transforms.Resize([256,256]),
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor()
    ]
)  #1
train_dataset = torchvision.datasets.ImageFolder(
    data_path,
    transform = transform
)  #2
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = 32,
    num_workers = 8,
    shuffle = True
)  #3

print(len(train_dataset))

385


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


1)  
`torchvision.transform`: 이미지 데이터 변환, 네트워크 입력으로 사용할 수 있게 변환  

2)  
`datasets.lmageFolder`: 데이터를 불러올 대상(경로), 방법(전처리) 정의

3)  
앞에서 정의한 ImageFolder를 데이터로더에 할당.  
- batch_size: 한 번에 불러올 데이터양 결정
- shuffle: 데이터 무작위로 섞을지를 지정  

[RandomResizedCrop]  



